In [1]:
import numpy as np 
import pandas as pd 


In [2]:
df = pd.read_csv('IMDBDataset_small.csv')
# df = pd.read_csv('IMDBDataset_small.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


Text preprocessing on the review

## LowerCasing

In [3]:
df['review'][0].lower()

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.<br /><br />it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />i would say the main appeal of the show is due to the fa

In [4]:
df['review'] = df['review'].str.lower()


In [5]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [6]:
df['review'][3]

"basically there's a family where a little boy (jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />this movie is slower than a soap opera... and suddenly, jake decides to become rambo and kill the zombie.<br /><br />ok, first of all when you're going to make a film you must decide if its a thriller or a drama! as a drama the movie is watchable. parents are divorcing & arguing like in real life. and then we have jake with his closet which totally ruins all the film! i expected to see a boogeyman similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. as for the shots with jake: just ignore them."

## Removing html if present

In [7]:
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'',str(text))

df['review'] = df['review'].apply(remove_html_tags)

Removing url if present

In [8]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', str(text))

df['review'] = df['review'].apply(remove_url)


Removing punctuation

In [9]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [10]:
exclude = string.punctuation

In [11]:
df['review'][1]


'a wonderful little production. the filming technique is very unassuming- very old-time-bbc fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. the actors are extremely well chosen- michael sheen not only "has got all the polari" but he has all the voices down pat too! you can truly see the seamless editing guided by the references to williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. a masterful production about one of the great master\'s of comedy and his life. the realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. it plays on our knowledge and our senses, particularly with the scenes concerning orton and halliwell and the sets (particularly of their flat with halliwell\'s murals decorating every surface) are terribly well done.'

In [12]:
def remove_punc_fast(text):
    return text.translate(str.maketrans('', '', exclude))

df['review'] = df['review'].apply(remove_punc_fast)


In [13]:
df['review'][1]

'a wonderful little production the filming technique is very unassuming very oldtimebbc fashion and gives a comforting and sometimes discomforting sense of realism to the entire piece the actors are extremely well chosen michael sheen not only has got all the polari but he has all the voices down pat too you can truly see the seamless editing guided by the references to williams diary entries not only is it well worth the watching but it is a terrificly written and performed piece a masterful production about one of the great masters of comedy and his life the realism really comes home with the little things the fantasy of the guard which rather than use the traditional dream techniques remains solid then disappears it plays on our knowledge and our senses particularly with the scenes concerning orton and halliwell and the sets particularly of their flat with halliwells murals decorating every surface are terribly well done'

## ChatWord Conversion 

Not needed here but might be needed mainly on the messages kind of data

## MisSpelled Treatment

In [14]:
from textblob import TextBlob

def fix_spelling(text):
    textBlb = TextBlob(text)

    return textBlb.correct().string

fix_spelling('ceertain')


'certain'

In [15]:
# df['review'] = df['review'].apply(fix_spelling)

## Removing Stop words 

In [16]:
from nltk.corpus import stopwords

In [17]:
def remove_stopwords(text):
    new_text = []
    
    for word in text.split():
        if word in stopwords.words('english'):
            new_text.append('')
        else:
            new_text.append(word)
    x = new_text[:]
    new_text.clear()
    return " ".join(x)

In [18]:
# df['review'] = df['review'].apply(remove_stopwords)
stop_words = set(stopwords.words("english"))

df["review"] = (
    df["review"]
    .fillna("")
    .str.lower()
    .str.split()
    .apply(lambda words: " ".join(
        word for word in words
        if word not in stop_words
    ))
)


In [19]:
df.sample()

,review,sentiment
351,never ever even consider watch movie absolutel...,negative


## Tokenizations 

using spacy for tokenization

In [20]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [21]:
def tokenize(text):
    docs = nlp(text)
    tokens =[]
    for token in docs:
        tokens.append(token)
    return tokens

# print(type(tokenize("Hello what's my name ?")))
print(tokenize("Hello what's my name ?"))
# def tokenize(text):
#     return [token.text for token in nlp.tokenizer(text)]

[Hello, what, 's, my, name, ?]


In [22]:
df['tokens'] = df['review'].apply(tokenize)


In [23]:
df.sample()
type(df['tokens'][0])

list

## Stemming 


In [24]:
from nltk.stem.porter import PorterStemmer # SnowballStemmer for the Other languages this works well for the english
ps = PorterStemmer()


In [25]:

def stem_words(tokens):
    # 1. Guard against rows that aren't lists (like NaN or float)
    if not isinstance(tokens, (list, tuple)):
        return []

    # 2. Stringify tokens and skip empty/null items to prevent the .lower() bug
    return [
        ps.stem(str(token))
        for token in tokens
        if token is not None and str(token).strip() != ""
    ]


# Apply the safe function
df["tokens"] = df["tokens"].apply(stem_words)

    # return " ".join([ps.stem(word) for word in text.split()])
test = [ "do ", "does ", "did", "doing" ]
stem_words(test)

['do ', 'does ', 'did', 'do']

In [26]:
df['tokens'] = df['tokens'].apply(stem_words)


In [27]:
df.sample()

,review,sentiment,tokens
221,think movie got really cool music never get he...,positive,"[think, movi, got, realli, cool, music, never,..."


## Lemmitizations 

Using WordNet to find the lemma (a word that always exit in language)

Using WordNet from nltk

In [28]:
import nltk
from nltk.stem import WordNetLemmatizer
wordnet_lemmatizer = WordNetLemmatizer()
def lemmatize(tokens):
    
    return [wordnet_lemmatizer.lemmatize(word,pos='v') for word in tokens]

test = ['the', 'mice', 'is', 'running', 'quickly', 'and', 'sees', 'two', 'building', '.']
lemmatize(test)

['the', 'mice', 'be', 'run', 'quickly', 'and', 'see', 'two', 'build', '.']

In [29]:
df['tokens'] = df['tokens'].apply(lemmatize)


In [30]:
df.sample()

,review,sentiment,tokens
234,three main characters hopeless yet feel sorry ...,positive,"[three, main, charact, hopeless, yet, feel, so..."


# Text Representation Implementation

In [31]:
# Problem 2

# Find out the number of words in the entire corpus and also the total number of unique words(vocabulary) using just python

In [32]:
corpus_count = 0
corpus = []
for i in df['tokens']:
    # print(i)
    corpus_count+=len(i)
    corpus.extend(i)



# uniques

vocabulary = set(corpus)




In [39]:
print(corpus_count)
# print(vocabulary)
print(len(vocabulary))

63115
10575


In [34]:
# Problem 3

# Apply One Hot Encoding

from sklearn.preprocessing import OneHotEncoder

words = np.array([
    ["I"],
    ["love"],
    ["NLP"],
    ["Python"]
])

encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(words)

print(encoder.categories_)
print(encoded)

[array(['I', 'NLP', 'Python', 'love'], dtype='<U6')]
[[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]


In [35]:
# Problem 4

# Apply bag words and find the vocabulary also find the times each word has occured

from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "I love NLP",
    "I love Python",
    "Python is useful"
]

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(documents)

print(vectorizer.get_feature_names_out())
print(X.toarray())


['is' 'love' 'nlp' 'python' 'useful']
[[0 1 1 0 0]
 [0 1 0 1 0]
 [1 0 0 1 1]]


In [40]:
X = vectorizer.fit_transform(df['review'])

print(vectorizer.get_feature_names_out())
print(X.toarray())

['007' '02' '0510' ... 'zulu' 'zwick' 'ísnt']
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


['007' '02' '0510' ... 'zulu' 'zwick' 'ísnt']


In [36]:
# Problem 5

# Apply bag of bi-gram and bag of tri-gram and write down your observation about the dimensionality of the vocabulary

from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "the movie is good",
    "the movie is not good"
]

# Unigram model
unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))
X_uni = unigram_vectorizer.fit_transform(documents)

# Unigram + bigram model
ngram_vectorizer = CountVectorizer(ngram_range=(1, 2))
X_ngram = ngram_vectorizer.fit_transform(documents)

print("Unigrams:")
print(unigram_vectorizer.get_feature_names_out())

print("\nUnigrams and bigrams:")
print(ngram_vectorizer.get_feature_names_out())

Unigrams:
['good' 'is' 'movie' 'not' 'the']

Unigrams and bigrams:
['good' 'is' 'is good' 'is not' 'movie' 'movie is' 'not' 'not good' 'the'
 'the movie']


In [37]:
# Problem 6

# Apply tf-idf and find out the idf scores of words, also find out the vocabulary.

from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "cat sat mat",
    "cat ate fish",
    "dog ate fish"
]

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(documents)

print("Vocabulary:")
print(vectorizer.get_feature_names_out())

print("\nTF-IDF matrix:")
print(X.toarray())

Vocabulary:
['ate' 'cat' 'dog' 'fish' 'mat' 'sat']

TF-IDF matrix:
[[0.         0.4736296  0.         0.         0.62276601 0.62276601]
 [0.57735027 0.57735027 0.         0.57735027 0.         0.        ]
 [0.51785612 0.         0.68091856 0.51785612 0.         0.        ]]


In [42]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X = vectorizer.fit_transform(df['review'])

print(X.shape)

(499, 5000)


In [43]:
print("Vocabulary:")
print(vectorizer.get_feature_names_out())

print("\nTF-IDF matrix:")
print(X.toarray())

Vocabulary:
['10' '100' '1000' ... 'zombies' 'zone' 'zooms']

TF-IDF matrix:
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
